# ASL Hackathon — Track 2: MediaPipe Hand Landmarks + MLP

Goal: a lightweight, robust model that works on **real webcams** for the Phase-2 live demo, and still puts up a competitive Kaggle score.

**Pipeline**
1. For each training image, run MediaPipe Hands → 21 (x, y, z) keypoints.
2. Normalize landmarks relative to the wrist (translation + scale invariant).
3. Train a small MLP (63 → 256 → 256 → 128 → 29) on the landmark vectors.
4. At inference: if MediaPipe finds no hand → predict `nothing`. Else run the MLP.

**Why this matters for Phase 2:** the CNN track trains on Kaggle's clean, uniform-background images and tends to degrade on real webcams. A landmark-based classifier is largely invariant to background and lighting, so the live demo behaves far better. We also get a tiny model (<1 MB) that can ship to the browser via ONNX.

**Before running:** add the Kaggle dataset `grassknoted/asl-alphabet` (`+ Add Data`). Update `TEST_DIR` when the hackathon test set drops. GPU not strictly needed (MediaPipe is CPU), but enable it anyway — the MLP training piece will use it.

In [ ]:
# MediaPipe is usually pre-installed on Kaggle; if not, this brings it in.
!pip install -q mediapipe joblib

In [ ]:
import os, gc, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

import mediapipe as mp
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('mediapipe', mp.__version__)

In [ ]:
# ---------------- Config ----------------
from collections import deque

CLASSES = [chr(c) for c in range(ord('A'), ord('Z') + 1)] + ['space', 'del', 'nothing']
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
NOTHING_IDX = CLASS_TO_IDX['nothing']

def find_dir_with_classes(base=Path('/kaggle/input'), max_depth=5):
    """BFS through /kaggle/input looking for a directory that contains both `A/` and `nothing/` subdirs."""
    if not base.exists():
        return None
    queue = deque([(base, 0)])
    while queue:
        d, depth = queue.popleft()
        try:
            if (d / 'A').is_dir() and (d / 'nothing').is_dir():
                return d
        except (OSError, PermissionError):
            pass
        if depth < max_depth:
            try:
                for c in d.iterdir():
                    if c.is_dir():
                        queue.append((c, depth + 1))
            except (OSError, PermissionError):
                pass
    return None

DATA_DIR = find_dir_with_classes()
if DATA_DIR is None:
    print('Contents of /kaggle/input:')
    for p in Path('/kaggle/input').rglob('*'):
        if p.is_dir():
            print(' ', p)
    raise FileNotFoundError('Could not auto-locate training data. Set DATA_DIR manually using the listing above.')
print(f'DATA_DIR = {DATA_DIR}')

# Try to auto-find a test folder. Override TEST_DIR by hand once the hackathon test set drops.
TEST_DIR = None
for cand in Path('/kaggle/input').rglob('*test*'):
    if not cand.is_dir():
        continue
    try:
        if any(p.suffix.lower() in {'.jpg', '.jpeg', '.png'} for p in cand.iterdir() if p.is_file()):
            TEST_DIR = cand
            break
    except (OSError, PermissionError):
        continue
print(f'TEST_DIR = {TEST_DIR}  (override manually if wrong)')

OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

CFG = dict(
    max_per_class=None,       # set to e.g. 1500 for a fast pilot run, None for full dataset
    min_detection_confidence=0.3,
    seed=42,
    epochs=80,
    batch_size=1024,
    lr=1e-3,
    weight_decay=1e-4,
    val_fraction=0.1,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG['seed'])
print('Device:', DEVICE)


In [ ]:
def scan_train(root: Path, max_per_class=None):
    rows = []
    rng = random.Random(CFG['seed'])
    for cls in CLASSES:
        cls_dir = root / cls
        if not cls_dir.is_dir():
            raise FileNotFoundError(f'Missing class dir: {cls_dir}')
        paths = [p for p in cls_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
        if max_per_class is not None and len(paths) > max_per_class:
            rng.shuffle(paths)
            paths = paths[:max_per_class]
        for p in paths:
            rows.append({'path': str(p), 'label': CLASS_TO_IDX[cls], 'class': cls})
    return pd.DataFrame(rows)

df = scan_train(DATA_DIR, max_per_class=CFG['max_per_class'])
print('Samples to process:', len(df))
print(df['class'].value_counts().sort_index())

In [ ]:
# ---------------- Landmark extraction ----------------
# Runs once, caches to /kaggle/working/landmarks.npz so reruns are instant.

mp_hands = mp.solutions.hands

def normalize_landmarks(pts: np.ndarray) -> np.ndarray:
    # pts: (21, 3) in image-relative coords. Center on wrist, scale by max dist to wrist.
    pts = pts - pts[0:1]
    scale = np.linalg.norm(pts, axis=1).max() + 1e-6
    return (pts / scale).flatten().astype(np.float32)

def extract_landmarks(img_bgr, hands_detector):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    res = hands_detector.process(img_rgb)
    if not res.multi_hand_landmarks:
        return None
    lm = res.multi_hand_landmarks[0]
    pts = np.array([[p.x, p.y, p.z] for p in lm.landmark], dtype=np.float32)
    return normalize_landmarks(pts)

cache_path = OUT_DIR / 'landmarks.npz'
if cache_path.exists():
    print(f'Loading cached landmarks from {cache_path}')
    cached = np.load(cache_path)
    X_all, y_all, detected = cached['X'], cached['y'], cached['detected']
else:
    X_list, y_list, det_list = [], [], []
    t0 = time.time()
    with mp_hands.Hands(static_image_mode=True, max_num_hands=1,
                        min_detection_confidence=CFG['min_detection_confidence']) as hands:
        for path, label in tqdm(zip(df['path'].values, df['label'].values), total=len(df)):
            img = cv2.imread(path)
            if img is None:
                X_list.append(np.zeros(63, dtype=np.float32)); det_list.append(0)
            else:
                lm = extract_landmarks(img, hands)
                if lm is None:
                    X_list.append(np.zeros(63, dtype=np.float32)); det_list.append(0)
                else:
                    X_list.append(lm); det_list.append(1)
            y_list.append(label)
    X_all = np.stack(X_list)
    y_all = np.array(y_list, dtype=np.int64)
    detected = np.array(det_list, dtype=np.int64)
    np.savez_compressed(cache_path, X=X_all, y=y_all, detected=detected)
    print(f'Extraction took {time.time()-t0:.1f}s')

print(f'Shape: X={X_all.shape}  y={y_all.shape}')
print(f'Overall detection rate: {detected.mean():.2%}')

# Per-class detection rate (low rate = MediaPipe struggles on this sign)
rates = pd.DataFrame({'class': [IDX_TO_CLASS[int(l)] for l in y_all], 'det': detected})
print('\nPer-class detection rate:')
print(rates.groupby('class')['det'].mean().sort_values().to_string())

In [ ]:
# ---------------- Training data prep ----------------
# Strategy: keep all detected samples. For 'nothing' specifically, keep both
# detected (rare) and undetected samples — at inference time, undetected hand
# => 'nothing', so the classifier mainly needs to discriminate the 28 letters.

is_nothing = (y_all == NOTHING_IDX)
keep_mask = (detected == 1) | is_nothing
X = X_all[keep_mask]
y = y_all[keep_mask]
print(f'Using {len(X)} samples ({(~keep_mask).sum()} dropped — no hand detected on non-nothing class)')

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=CFG['val_fraction'], stratify=y, random_state=CFG['seed']
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
print(f'train={len(X_train_s)}  val={len(X_val_s)}')

In [ ]:
class LandmarkMLP(nn.Module):
    def __init__(self, in_dim=63, num_classes=29):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.GELU(),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.net(x)

model = LandmarkMLP(63, NUM_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'MLP params: {n_params:,}  (~{n_params*4/1e6:.2f} MB fp32)')

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.03)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])

X_train_t = torch.from_numpy(X_train_s).to(DEVICE)
y_train_t = torch.from_numpy(y_train).long().to(DEVICE)
X_val_t = torch.from_numpy(X_val_s).to(DEVICE)
y_val_t = torch.from_numpy(y_val).long().to(DEVICE)

best_acc = 0.0
ckpt_path = OUT_DIR / 'landmark_mlp_best.pth'
scaler_path = OUT_DIR / 'landmark_scaler.pkl'
history = []

for epoch in range(CFG['epochs']):
    model.train()
    perm = torch.randperm(len(X_train_t), device=DEVICE)
    losses = []
    for i in range(0, len(perm), CFG['batch_size']):
        idx = perm[i:i + CFG['batch_size']]
        optimizer.zero_grad(set_to_none=True)
        out = model(X_train_t[idx])
        loss = criterion(out, y_train_t[idx])
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()
    model.eval()
    with torch.no_grad():
        val_out = model(X_val_t)
        val_acc = (val_out.argmax(1) == y_val_t).float().mean().item()
        val_loss = criterion(val_out, y_val_t).item()
    history.append({'epoch': epoch + 1, 'train_loss': float(np.mean(losses)), 'val_loss': val_loss, 'val_acc': val_acc})
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({'state_dict': model.state_dict(), 'classes': CLASSES, 'val_acc': val_acc}, ckpt_path)
        joblib.dump(scaler, scaler_path)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}: train_loss={np.mean(losses):.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  (best={best_acc:.4f})')

pd.DataFrame(history).to_csv(OUT_DIR / 'history_landmark.csv', index=False)
print(f'\nBest val acc: {best_acc:.4f}')
print(f'Checkpoint: {ckpt_path}')
print(f'Scaler:     {scaler_path}')

In [ ]:
# Confusion-style report on val set — which signs is the model confusing?
from collections import Counter
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)['state_dict'])
model.eval()
with torch.no_grad():
    val_pred = model(X_val_t).argmax(1).cpu().numpy()
val_true = y_val

per_class = {}
for cls_idx, cls_name in IDX_TO_CLASS.items():
    mask = (val_true == cls_idx)
    if mask.sum() == 0: continue
    acc = (val_pred[mask] == cls_idx).mean()
    per_class[cls_name] = acc
print('Per-class val accuracy (worst first):')
for cls, acc in sorted(per_class.items(), key=lambda kv: kv[1])[:10]:
    print(f'  {cls:>8s}: {acc:.3f}')

print('\nTop confusions:')
confusions = Counter()
for t, p in zip(val_true, val_pred):
    if t != p:
        confusions[(IDX_TO_CLASS[int(t)], IDX_TO_CLASS[int(p)])] += 1
for (t, p), n in confusions.most_common(10):
    print(f'  {t} -> {p}: {n}')

In [ ]:
# Export the MLP to ONNX so we can ship it to the browser (Phase 2 live demo).
onnx_path = OUT_DIR / 'landmark_mlp.onnx'
dummy = torch.zeros(1, 63, device=DEVICE)
torch.onnx.export(
    model, dummy, str(onnx_path),
    input_names=['landmarks'], output_names=['logits'],
    dynamic_axes={'landmarks': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
)
print(f'ONNX written to {onnx_path}  ({onnx_path.stat().st_size/1024:.1f} KB)')

# Also save scaler params as JSON so the JS side can match preprocessing exactly
scaler_json = {'mean': scaler.mean_.tolist(), 'scale': scaler.scale_.tolist()}
(OUT_DIR / 'landmark_scaler.json').write_text(json.dumps(scaler_json))
print('Scaler params -> landmark_scaler.json')

In [ ]:
# ---------------- Submission ----------------
def stem_to_id(stem: str) -> str:
    # Customise if the hackathon expects a different image_id format.
    return stem

def predict_image(img_bgr, hands_detector):
    lm = extract_landmarks(img_bgr, hands_detector)
    if lm is None:
        return 'nothing'
    x = scaler.transform(lm.reshape(1, -1)).astype(np.float32)
    with torch.no_grad():
        out = model(torch.from_numpy(x).to(DEVICE))
        idx = int(out.argmax(1).item())
    return IDX_TO_CLASS[idx]

if TEST_DIR.exists():
    test_paths = sorted([p for p in TEST_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
    print(f'Predicting {len(test_paths)} test images from {TEST_DIR}')
    rows = []
    with mp_hands.Hands(static_image_mode=True, max_num_hands=1,
                        min_detection_confidence=CFG['min_detection_confidence']) as hands:
        for p in tqdm(test_paths):
            img = cv2.imread(str(p))
            if img is None:
                label = 'nothing'
            else:
                label = predict_image(img, hands)
            rows.append({'image_id': stem_to_id(p.stem), 'label': label})
    sub = pd.DataFrame(rows)
    sub_path = OUT_DIR / 'submission.csv'
    sub.to_csv(sub_path, index=False)
    print(sub.head())
    print(f'Wrote {sub_path}')
else:
    print(f'Test dir not found: {TEST_DIR}')
    print('Update TEST_DIR in the config cell when the hackathon test set is released.')

## Notes

- **Time budget:** landmark extraction on 87k images takes 30–60 min single-threaded. After that, MLP training is < 1 min. Re-runs of this notebook reuse the cached `landmarks.npz`.
- **Quick pilot:** set `max_per_class=1500` in the config cell for a ~2-minute end-to-end sanity check.
- **Detection rate is a Phase-2 health signal:** if a class has very low MediaPipe detection rate, the live webcam demo will struggle on that sign. Watch the per-class table.
- **Outputs ready for Phase 2:**
  - `landmark_mlp.onnx` — ship to browser via `onnxruntime-web`.
  - `landmark_scaler.json` — JS preprocessing must apply this normalization before inference.
- **Don't expect to beat the CNN on the leaderboard** — this track exists to power the live demo (25% of judging) and as a robustness sanity check.